In [ ]:
import pandas as pd
import math
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_parquet("/Users/yuliia/Documents/diploma/parguet/processed_transactions.parquet")

frauds = [11968000, 11970409, 11726701, 14827913, 12411311, 11098795, 12412748, 11812494, 12396334]

client_data = pd.read_parquet("/Users/yuliia/Documents/diploma/parguet/client_level_features.parquet") 

## Visualizations

### Client
на основі даних клієнта [person_id]

In [ ]:
features = [
    'person_id',
    'is_fraud',
    'num_of_trn', 
    'days_visits', 
    'gross_amount_mean', 
    'gross_amount_sum',
    'bonuses_accum_sum',
    'bonuses_used_sum',
    'num_of_waiters',
    'gross_amount_max',
    'first_last_trn_diff',
    'first_second_trn_diff',
    'first_third_trn_diff',
    'time_between_trn_median',
    'trn_per_day',
    'share_top_waiter',
    'share_bonus_trn',
    'share_bonus_after_first'
    ]

In [ ]:
rename_dict = {
    "num_of_trn": "Number of Transactions",
    "days_visits": "Visit Days",
    "gross_amount_mean": "Avg Check Amount",
    "gross_amount_sum": "Total Spend",
    "bonuses_accum_sum": "Total Bonuses Accumulated",
    "bonuses_used_sum": "Total Bonuses Used",
    'num_of_waiters': "Number of Waiters Interacted With",
    'first_last_trn_diff': "Hours Between First and Last Transaction",
    'gross_amount_max': "Max Check Amount",
    'first_second_trn_diff': "Hours Between First and Second Transaction",
    'first_third_trn_diff': "Hours Between First and Third Transaction",
    'time_between_trn_median': "Median Time Between Transactions (Hours)",
    'trn_per_day': "Transactions Per Day",
    'share_top_waiter': "Share of Transactions with Top Waiter",
    'share_bonus_trn': "Share of Transactions with Bonuses Used",
    'share_bonus_after_first': "Share of Transactions with Bonuses Used After First Transaction"
}

In [ ]:

features_to_plot = [f for f in features if f not in ['person_id', 'is_fraud']]
n_cols = 3
n_rows = math.ceil(len(features_to_plot) / n_cols)

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[rename_dict.get(f, f) for f in features_to_plot],
    horizontal_spacing=0.03,
    vertical_spacing=0.03,
)

log_features = {
    'num_of_trn','days_visits','gross_amount_mean','gross_amount_sum',
    'bonuses_accum_sum','bonuses_used_sum','num_of_waiters','gross_amount_max',
    'first_last_trn_diff','first_second_trn_diff','first_third_trn_diff',
    'time_between_trn_median','trn_per_day'
}

DROP_ZEROS_FOR_LOG = False

for i, feat in enumerate(features_to_plot):
    r = i // n_cols + 1
    c = i % n_cols + 1

    normal = client_data.loc[~client_data['is_fraud'], feat].astype(float)
    fraud  = client_data.loc[ client_data['is_fraud'], feat].astype(float)

    use_log = feat in log_features
    if use_log and DROP_ZEROS_FOR_LOG:
        normal = normal.where(normal > 0)
        fraud  = fraud.where(fraud > 0)

    fig.add_trace(
        go.Box(
            y=normal,
            name="Normal",
            marker_color="#636EFA",
            boxpoints=False,
            width=0.6,
            quartilemethod="linear",
            showlegend=(i == 0),
            legendgroup="Normal",
        ),
        row=r, col=c
    )

    fig.add_trace(
        go.Box(
            y=fraud,
            name="Fraud",
            marker_color="#EF553B",
            boxpoints=False,
            width=0.6,
            quartilemethod="linear",
            showlegend=(i == 0),
            legendgroup="Fraud",
        ),
        row=r, col=c
    )

    if use_log:
        fig.update_yaxes(type="log", row=r, col=c)

    fig.update_xaxes(showgrid=False, row=r, col=c)
    fig.update_yaxes(showgrid=True, gridcolor="rgba(200,200,200,0.3)", row=r, col=c)

fig.update_layout(
    title="Client Behaviour Distribution (Normal vs Fraud)",
    title_x=0.5,
    height=500 * n_rows,
    width=1800,
    boxmode="group",
    template="plotly_white",
    font=dict(size=16),
)

fig.show()

## for "active" users with 2+ trn

In [ ]:
active_clients = client_data[client_data['num_of_trn'] > 1]

In [ ]:
import math
import plotly.graph_objects as go
from plotly.subplots import make_subplots

features_to_plot = [f for f in features if f not in ['person_id', 'is_fraud']]
n_cols = 3
n_rows = math.ceil(len(features_to_plot) / n_cols)

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[rename_dict.get(f, f) for f in features_to_plot],
    horizontal_spacing=0.03,
    vertical_spacing=0.03,
)

log_features = {
    'num_of_trn','days_visits','gross_amount_mean','gross_amount_sum',
    'bonuses_accum_sum','bonuses_used_sum','num_of_waiters','gross_amount_max',
    'first_last_trn_diff','first_second_trn_diff','first_third_trn_diff',
    'time_between_trn_median','trn_per_day'
}

DROP_ZEROS_FOR_LOG = False

for i, feat in enumerate(features_to_plot):
    r = i // n_cols + 1
    c = i % n_cols + 1

    normal = active_clients.loc[~active_clients['is_fraud'], feat].astype(float)
    fraud  = active_clients.loc[ active_clients['is_fraud'], feat].astype(float)

    use_log = feat in log_features
    if use_log and DROP_ZEROS_FOR_LOG:
        normal = normal.where(normal > 0)
        fraud  = fraud.where(fraud > 0)

    fig.add_trace(
        go.Box(
            y=normal,
            name="Normal",
            marker_color="#636EFA",
            boxpoints=False,
            width=0.6,
            quartilemethod="linear",
            showlegend=(i == 0),
            legendgroup="Normal",
        ),
        row=r, col=c
    )

    fig.add_trace(
        go.Box(
            y=fraud,
            name="Fraud",
            marker_color="#EF553B",
            boxpoints=False,
            width=0.6,
            quartilemethod="linear",
            showlegend=(i == 0),
            legendgroup="Fraud",
        ),
        row=r, col=c
    )

    if use_log:
        fig.update_yaxes(type="log", row=r, col=c)

    fig.update_xaxes(showgrid=False, row=r, col=c)
    fig.update_yaxes(showgrid=True, gridcolor="rgba(200,200,200,0.3)", row=r, col=c)

fig.update_layout(
    title="Client Behaviour Distribution (Normal vs Fraud)",
    title_x=0.5,
    height=500 * n_rows,
    width=1800,
    boxmode="group",
    template="plotly_white",
    font=dict(size=16),
)

fig.show()